# 05 - Statistical Significance Testing

Testing whether observed performance differences (between models, and between input representations) are statistically meaningful, rather than relying on visual comparison of means and error bars.

**Method:** paired Wilcoxon signed-rank test, across the 5 matched cross-validation folds.

**Two families of comparisons:**
1. **Model vs. model, within the same input** (e.g. Coxnet vs. RSF, both on single-omics) - is one model significantly better than another?
2. **Input vs. input, within the same model** (e.g. single-omics vs. MOFA, both using RSF) - is one integration method significantly better than another?

**Metrics tested:** C-index (higher is better) and IBS (lower is better, NaN folds automatically excluded from the pairing).



## 0. Set working directory

In [9]:
import os
os.chdir(r"PATH_TO_YOUR_DIRECTORY")  # Change this to your desired directory
print(os.getcwd())


FileNotFoundError: [WinError 2] The system cannot find the file specified: 'PATH_TO_YOUR_DIRECTORY'

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
import common_utils as cu


## 2. Load all three experiments' results

In [ ]:
experiments = {
    "Single-omics": cu.load_results("expression"),
    "Concatenated": cu.load_results("concatenated"),
    "MOFA": cu.load_results("mofa"),
}
models = ["coxnet", "rsf", "xgboost"]


## 3. Helper: paired Wilcoxon test with NaN-safe pairing

For IBS, some folds may be NaN (e.g. Elastic-Net Cox's numerical instability on certain folds). We only compare folds where BOTH sides of the pair have a valid value.

In [ ]:
def paired_wilcoxon(values_a, values_b, higher_is_better=True):
    a = np.array(values_a, dtype=float)
    b = np.array(values_b, dtype=float)
    valid = ~np.isnan(a) & ~np.isnan(b)
    n_valid = valid.sum()

    if n_valid < 3:
        return {"n_pairs": n_valid, "mean_diff": np.nan, "p_value": np.nan, "note": "too few valid pairs"}

    a_valid, b_valid = a[valid], b[valid]
    diff = a_valid - b_valid if higher_is_better else b_valid - a_valid

    if np.all(diff == 0):
        return {"n_pairs": n_valid, "mean_diff": 0.0, "p_value": 1.0, "note": "identical values"}

    try:
        stat, p = wilcoxon(a_valid, b_valid)
    except ValueError as e:
        return {"n_pairs": n_valid, "mean_diff": np.mean(diff), "p_value": np.nan, "note": str(e)}

    return {"n_pairs": n_valid, "mean_diff": np.mean(diff), "p_value": p, "note": ""}


## 4. Model vs. model, within each input

For each input representation, compare every pair of models on C-index and IBS.

In [ ]:
from itertools import combinations

rows = []
for exp_name, res in experiments.items():
    for model_a, model_b in combinations(models, 2):
        c_result = paired_wilcoxon(res[model_a]["c_index"], res[model_b]["c_index"], higher_is_better=True)
        ibs_result = paired_wilcoxon(res[model_a]["ibs"], res[model_b]["ibs"], higher_is_better=False)

        rows.append({
            "comparison_type": "model vs model",
            "context": exp_name,
            "A": model_a, "B": model_b,
            "metric": "C-index",
            "n_pairs": c_result["n_pairs"],
            "mean_diff_A_minus_B": c_result["mean_diff"],
            "p_value": c_result["p_value"],
        })
        rows.append({
            "comparison_type": "model vs model",
            "context": exp_name,
            "A": model_a, "B": model_b,
            "metric": "IBS",
            "n_pairs": ibs_result["n_pairs"],
            "mean_diff_A_minus_B": ibs_result["mean_diff"],
            "p_value": ibs_result["p_value"],
        })

model_vs_model = pd.DataFrame(rows)
model_vs_model


,comparison_type,context,A,B,metric,n_pairs,mean_diff_A_minus_B,p_value
0,model vs model,Single-omics,coxnet,rsf,C-index,5,-0.047575,0.1875
1,model vs model,Single-omics,coxnet,rsf,IBS,4,-0.056226,0.1250
2,model vs model,Single-omics,coxnet,xgboost,C-index,5,-0.039965,0.3125
3,model vs model,Single-omics,coxnet,xgboost,IBS,4,-0.031309,0.3750
4,model vs model,Single-omics,rsf,xgboost,C-index,5,0.007610,0.6250
5,model vs model,Single-omics,rsf,xgboost,IBS,5,0.025065,0.0625
6,model vs model,Concatenated,coxnet,rsf,C-index,5,-0.068205,0.1250
7,model vs model,Concatenated,coxnet,rsf,IBS,2,NaN,NaN
8,model vs model,Concatenated,coxnet,xgboost,C-index,5,-0.075707,0.1250
9,model vs model,Concatenated,coxnet,xgboost,IBS,2,NaN,NaN


## 5. Input vs. input, within each model

For each model, compare every pair of input representations on C-index and IBS.

In [ ]:
rows = []
exp_names = list(experiments.keys())
for model in models:
    for exp_a, exp_b in combinations(exp_names, 2):
        c_result = paired_wilcoxon(experiments[exp_a][model]["c_index"], experiments[exp_b][model]["c_index"], higher_is_better=True)
        ibs_result = paired_wilcoxon(experiments[exp_a][model]["ibs"], experiments[exp_b][model]["ibs"], higher_is_better=False)

        rows.append({
            "comparison_type": "input vs input",
            "context": model,
            "A": exp_a, "B": exp_b,
            "metric": "C-index",
            "n_pairs": c_result["n_pairs"],
            "mean_diff_A_minus_B": c_result["mean_diff"],
            "p_value": c_result["p_value"],
        })
        rows.append({
            "comparison_type": "input vs input",
            "context": model,
            "A": exp_a, "B": exp_b,
            "metric": "IBS",
            "n_pairs": ibs_result["n_pairs"],
            "mean_diff_A_minus_B": ibs_result["mean_diff"],
            "p_value": ibs_result["p_value"],
        })

input_vs_input = pd.DataFrame(rows)
input_vs_input


,comparison_type,context,A,B,metric,n_pairs,mean_diff_A_minus_B,p_value
0,input vs input,coxnet,Single-omics,Concatenated,C-index,5,0.007408,1.0000
1,input vs input,coxnet,Single-omics,Concatenated,IBS,2,NaN,NaN
2,input vs input,coxnet,Single-omics,MOFA,C-index,5,-0.002884,0.8125
3,input vs input,coxnet,Single-omics,MOFA,IBS,4,-0.035953,0.3750
4,input vs input,coxnet,Concatenated,MOFA,C-index,5,-0.010291,0.8125
5,input vs input,coxnet,Concatenated,MOFA,IBS,2,NaN,NaN
6,input vs input,rsf,Single-omics,Concatenated,C-index,5,-0.013222,0.4375
7,input vs input,rsf,Single-omics,Concatenated,IBS,5,0.000488,1.0000
8,input vs input,rsf,Single-omics,MOFA,C-index,5,0.016770,0.6250
9,input vs input,rsf,Single-omics,MOFA,IBS,5,0.006851,0.0625


## 6. Combine, flag significance, and save

Using alpha = 0.05, uncorrected


In [ ]:
all_results = pd.concat([model_vs_model, input_vs_input], ignore_index=True)
all_results["significant_uncorrected"] = all_results["p_value"] < 0.05

n_tests = all_results["p_value"].notna().sum()
bonferroni_alpha = 0.05 / n_tests
all_results["significant_bonferroni"] = all_results["p_value"] < bonferroni_alpha

print(f"Total tests run: {n_tests}")
print(f"Bonferroni-corrected alpha: {bonferroni_alpha:.4f}")
print(f"Significant at uncorrected p<0.05: {all_results['significant_uncorrected'].sum()}")
print(f"Significant after Bonferroni correction: {all_results['significant_bonferroni'].sum()}")

all_results.to_csv("statistical_significance_results.csv", index=False)
all_results


Total tests run: 32
Bonferroni-corrected alpha: 0.0016
Significant at uncorrected p<0.05: 0
Significant after Bonferroni correction: 0


,comparison_type,context,A,B,metric,n_pairs,mean_diff_A_minus_B,p_value,significant_uncorrected,significant_bonferroni
0,model vs model,Single-omics,coxnet,rsf,C-index,5,-0.047575,0.1875,False,False
1,model vs model,Single-omics,coxnet,rsf,IBS,4,-0.056226,0.1250,False,False
2,model vs model,Single-omics,coxnet,xgboost,C-index,5,-0.039965,0.3125,False,False
3,model vs model,Single-omics,coxnet,xgboost,IBS,4,-0.031309,0.3750,False,False
4,model vs model,Single-omics,rsf,xgboost,C-index,5,0.007610,0.6250,False,False
5,model vs model,Single-omics,rsf,xgboost,IBS,5,0.025065,0.0625,False,False
6,model vs model,Concatenated,coxnet,rsf,C-index,5,-0.068205,0.1250,False,False
7,model vs model,Concatenated,coxnet,rsf,IBS,2,NaN,NaN,False,False
8,model vs model,Concatenated,coxnet,xgboost,C-index,5,-0.075707,0.1250,False,False
9,model vs model,Concatenated,coxnet,xgboost,IBS,2,NaN,NaN,False,False


## 7. Summary view: just the significant results (if any)

In [ ]:
significant_only = all_results[all_results["significant_uncorrected"]].sort_values("p_value")
if len(significant_only) == 0:
    print("No comparisons reached uncorrected significance (p < 0.05).")
    print("models/inputs are not statistically distinguishable given 5 paired folds.")
else:
    print(significant_only.to_string(index=False))


No comparisons reached uncorrected significance (p < 0.05).
This itself is a defensible, reportable finding: performance differences between
models/inputs are not statistically distinguishable given 5 paired folds.
